# Read combined DAWN revenue accounts
Load `data/dawn_revenue_accounts_combined_history.csv` into a pandas DataFrame named `df`. Run the cells in order. Requires pandas (`%pip install pandas` if needed).

Initial filter: **exclude GL Number 42** in the CSV-loading cell, before the preview, BA 9999 filter, and revenue calculations. The cell prints the number of removed rows and checks that no GL 42 remains.


In [1]:
from pathlib import Path
import pandas as pd

# Find the data folder from either the project root or notebook folder.
filename = 'dawn_revenue_accounts_combined_history.csv'
data_path = next(
    (folder / 'data' / filename
     for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / 'data' / filename).is_file()),
    None,
)
if data_path is None:
    raise FileNotFoundError(f'Could not find data/{filename}. Start Jupyter inside the project folder.')

print(f'Reading: {data_path}')

Reading: c:\Users\Syed.Uddin\OneDrive - State of Nevada\DAWN_DATA\dawn_aggregate\data\dawn_revenue_accounts_combined_history.csv


In [2]:
# Preserve account and document identifiers as text, including leading zeros.
identifier_columns = ['Doc Number', 'level_1_id', 'level_2_id', 'GL Number', 'BA Number']
df = pd.read_csv(
    data_path,
    dtype={column: 'string' for column in identifier_columns},
    low_memory=False,
)
# INITIAL FILTER: EXCLUDE GL NUMBER 42
# Keep rows where GL Number != '42', for every BA account and fiscal year.
# GL Number is text because it was loaded with dtype='string'.
# Strip surrounding spaces; retain rows with a missing GL Number.
rows_before_gl_filter = len(df)
keep_gl = (df['GL Number'].str.strip() != '42').fillna(True)
df = df.loc[keep_gl].copy()
print(f'GL 42 filter: removed {rows_before_gl_filter - len(df):,} rows; {len(df):,} rows remain.')
# Verify the exclusion whenever this cell runs.
assert not df['GL Number'].str.strip().eq('42').fillna(False).any(), 'GL 42 remains in df'

# RECODE GL 3290: classify as MAJOR for every BA account and fiscal year.
is_gl_3290 = df['GL Number'].str.strip().eq('3290').fillna(False)
df.loc[is_gl_3290, 'Type'] = 'MAJOR'
print(f'GL 3290 recode: {is_gl_3290.sum():,} rows classified as MAJOR.')

df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y', errors='raise')

print(f'Loaded {df.shape[0]:,} rows and {df.shape[1]} columns.')
df.head(10)




GL 42 filter: removed 5,673 rows; 328,684 rows remain.
GL 3290 recode: 578 rows classified as MAJOR.
Loaded 328,684 rows and 10 columns.


,Doc Number,Date,Amount,level_1_id,level_2_id,GL Number,description,Type,BA Number,fy
0,CR 010 00008235185,2019-07-29,200.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
1,CR 010 00008235187,2019-07-30,10724.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
2,CR 010 070619ZUFFA,2019-07-30,363822.43,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
3,CR 010 072519MGMTG,2019-07-30,421738.50,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
4,CR 010 00008235189,2019-08-12,752.80,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
5,CR 010 00008235190,2019-08-13,10200.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
6,CR 010 00008235191,2019-08-22,10400.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
7,CR 010 00008235192,2019-08-27,8613.40,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
8,CR 010 00008235193,2019-08-29,100.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
9,CR 010 00008235194,2019-09-06,2153.82,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020


In [3]:
print(df['Type'].unique())

<ArrowStringArray>
['Minor', 'Not Found in May 2025 forecast sheet', nan, 'MAJOR']
Length: 4, dtype: str


In [4]:
# FY 2026 revenue summary using df after the GL 42 and BA 9999 filters.
fy_2026 = df.loc[pd.to_numeric(df['fy'], errors='coerce').eq(2026)]
revenue_type = fy_2026['Type'].astype('string').str.strip()
major_revenue_2026 = fy_2026.loc[revenue_type.eq('MAJOR'), 'Amount'].sum()
minor_revenue_2026 = fy_2026.loc[revenue_type.eq('Minor'), 'Amount'].sum()
# Total includes all Types, including any missing or other Type values.
total_revenue_2026 = fy_2026['Amount'].sum()
revenue_summary_2026 = pd.DataFrame({
    'Category': ['Major', 'Minor', 'Total (all Types)'],
    'FY': [2026, 2026, 2026],
    'Total Revenue': [major_revenue_2026, minor_revenue_2026, total_revenue_2026],
})
revenue_summary_2026.style.format({'Total Revenue': '{:,.2f}'}).hide(axis='index')


Category,FY,Total Revenue
Major,2026,"5,487,700,774.87"
Minor,2026,"784,941,896.93"
Total (all Types),2026,"6,619,720,888.99"


In [5]:
# Restrict BA 9999 to the selected GL numbers; retain all other BAs.
allowed_gl_9999 = [
    '3107', '3114', '3115', '3120', '3202', '3203', '3204',
    '3220', '3271', '3274', '3276', '3291', '4102', '4403',
    '4408', '4409', '4621',
]
is_ba_9999 = df['BA Number'].str.strip().eq('9999').fillna(False)
is_allowed_gl = df['GL Number'].str.strip().isin(allowed_gl_9999)
rows_before = len(df)
df = df.loc[~is_ba_9999 | is_allowed_gl].copy()
print(f'Kept {len(df):,} rows; removed {rows_before - len(df):,} rows from BA 9999.')
df.head(10)


Kept 297,091 rows; removed 31,593 rows from BA 9999.


,Doc Number,Date,Amount,level_1_id,level_2_id,GL Number,description,Type,BA Number,fy
0,CR 010 00008235185,2019-07-29,200.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
1,CR 010 00008235187,2019-07-30,10724.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
2,CR 010 070619ZUFFA,2019-07-30,363822.43,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
3,CR 010 072519MGMTG,2019-07-30,421738.50,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
4,CR 010 00008235189,2019-08-12,752.80,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
5,CR 010 00008235190,2019-08-13,10200.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
6,CR 010 00008235191,2019-08-22,10400.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
7,CR 010 00008235192,2019-08-27,8613.40,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
8,CR 010 00008235193,2019-08-29,100.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
9,CR 010 00008235194,2019-09-06,2153.82,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020


In [6]:
# Inspect column types and non-null counts.
df.info()

<class 'pandas.DataFrame'>
Index: 297091 entries, 0 to 334346
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Doc Number   297091 non-null  string        
 1   Date         281195 non-null  datetime64[us]
 2   Amount       297091 non-null  float64       
 3   level_1_id   297091 non-null  string        
 4   level_2_id   297091 non-null  string        
 5   GL Number    297091 non-null  string        
 6   description  297091 non-null  str           
 7   Type         297090 non-null  str           
 8   BA Number    297091 non-null  string        
 9   fy           297091 non-null  int64         
dtypes: datetime64[us](1), float64(1), int64(1), str(2), string(5)
memory usage: 43.0 MB


In [7]:
# Total FY 2026 Minor revenue by BA account, after the BA 9999 filter.
minor_2026 = df.loc[
    df['Type'].eq('Minor') & pd.to_numeric(df['fy'], errors='coerce').eq(2026)
]
revenue_by_ba = (
    minor_2026.groupby('BA Number', as_index=False)['Amount']
    .sum(min_count=1)
    .rename(columns={'Amount': 'Total Revenue'})
    .sort_values('BA Number')
    .reset_index(drop=True)
)
revenue_by_ba.style.format({'Total Revenue': '{:,.2f}'}).hide(axis='index')


BA Number,Total Revenue
9010,"6,185,484.75"
9040,"266,597,544.15"
9090,"181,490.00"
9130,"382,882,134.48"
9360,"237,979.00"
9600,"50,980,520.86"
9705,"3,368,123.49"
9707,"1,081,184.06"
9741,"31,904,438.68"
9748,"3,707,035.46"


In [10]:
# FY 2026 MAJOR revenue by BA account.
# Uses df after the GL 42 exclusion, GL 3290 recode, and BA 9999 filter.
major_2026 = df.loc[
    df['Type'].astype('string').str.strip().str.upper().eq('MAJOR')
    & pd.to_numeric(df['fy'], errors='coerce').eq(2026)
]
major_revenue_by_ba = (
    major_2026.groupby('BA Number', as_index=False)['Amount']
    .sum(min_count=1)
    .rename(columns={'Amount': 'Total Major Revenue'})
    .sort_values('BA Number')
    .reset_index(drop=True)
)
major_revenue_by_ba.style.format({'Total Major Revenue': '{:,.2f}'}).hide(axis='index')


BA Number,Total Major Revenue
9050,"164,646,086.06"
9130,"4,165,755,541.03"
9600,"1,121,670,811.18"
9741,"35,628,336.60"


In [11]:
# FY 2026 MAJOR revenue by BA account.
# Uses df after the GL 42 exclusion, GL 3290 recode, and BA 9999 filter.
major_2026 = df.loc[
    df['Type'].astype('string').str.strip().str.upper().eq('MAJOR')
    & pd.to_numeric(df['fy'], errors='coerce').eq(2026)
]
major_revenue_by_ba = (
    major_2026.groupby('GL Number', as_index=False)['Amount']
    .sum(min_count=1)
    .rename(columns={'Amount': 'Total Major Revenue'})
    .sort_values('GL Number')
    .reset_index(drop=True)
)
major_revenue_by_ba.style.format({'Total Major Revenue': '{:,.2f}'}).hide(axis='index')


GL Number,Total Major Revenue
3001,"1,926,962,816.10"
3031,"216,726,025.86"
3041,"1,032,740,382.93"
3055,"121,027,132.76"
3061,"732,911,603.50"
3069,"810,980,705.96"
3072,"481,706,021.70"
3290,"164,646,086.06"


In [12]:
# FY 2026 MAJOR revenue by BA account.
# Uses df after the GL 42 exclusion, GL 3290 recode, and BA 9999 filter.
major_2026 = df.loc[
    df['Type'].astype('string').str.strip().str.upper().eq('MAJOR')
    & pd.to_numeric(df['fy'], errors='coerce').eq(2026)
]
major_revenue_by_ba = (
    major_2026.groupby('GL Number', as_index=False)['Amount']
    .sum(min_count=1)
    .rename(columns={'Amount': 'Total Major Revenue'})
    .sort_values('GL Number')
    .reset_index(drop=True)
)
major_revenue_by_ba.style.format({'Total Major Revenue': '{:,.2f}'}).hide(axis='index')


GL Number,Total Major Revenue
3001,"1,926,962,816.10"
3031,"216,726,025.86"
3041,"1,032,740,382.93"
3055,"121,027,132.76"
3061,"732,911,603.50"
3069,"810,980,705.96"
3072,"481,706,021.70"
3290,"164,646,086.06"
